In [1]:
!pip install datasets
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is i

In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 4.9 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch
from jiwer import wer
import evaluate


In [4]:
librispeech = load_dataset("RaphaelOlivier/librispeech_asr_adversarial", "adv", split='natural')

README.md:   0%|          | 0.00/2.65k [00:00<?, ?B/s]

librispeech_asr_adversarial.py:   0%|          | 0.00/5.59k [00:00<?, ?B/s]

The repository for RaphaelOlivier/librispeech_asr_adversarial contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/RaphaelOlivier/librispeech_asr_adversarial.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating natural split: 0 examples [00:00, ? examples/s]

Generating adv_0.04 split: 0 examples [00:00, ? examples/s]

Generating adv_0.015 split: 0 examples [00:00, ? examples/s]

Generating adv_0.015_RIR split: 0 examples [00:00, ? examples/s]

In [6]:
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
import IPython.display as ipd
example = librispeech[10]

audio_array = example['audio']['array']

display(ipd.Audio(audio_array, rate=16000))
print(example["true_text"])



IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [ ]:
from utils import transcribe_audio

In [9]:
predicted_transcription = transcribe_audio(audio_array, 16000, processor, model)
print(predicted_transcription)

IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [ ]:
from jiwer import wer
import evaluate
cer_metric = evaluate.load("cer")
import json


# Extract audio arrays, sampling rates, and ground truths
audio_arrays = [example["audio"]["array"] for example in librispeech]
sampling_rates = [example["audio"]["sampling_rate"] for example in librispeech]
ground_truths = [example["true_text"].lower().strip() for example in librispeech]

# Generate transcriptions for all samples
try:
    transcriptions = [
        transcribe_audio(audio_array, sampling_rate, processor, model).lower().strip()
        for audio_array, sampling_rate in zip(audio_arrays, sampling_rates)
    ]
except Exception as e:
    print(f"Error during batch transcription: {e}")
    transcriptions = []


In [ ]:
import evaluate

# load both metrics
wer_metric = evaluate.load("wer")    # Word‑Error‑Rate :contentReference[oaicite:0]{index=0}
cer_metric = evaluate.load("cer")


# compute them in one shot
avg_wer = wer_metric.compute(predictions=transcriptions, references=ground_truths)
avg_cer = cer_metric.compute(predictions=transcriptions, references=ground_truths)

print(f"Average WER: {avg_wer:.4f} ({avg_wer*100:.2f}%)")
print(f"Average CER: {avg_cer:.4f} ({avg_cer*100:.2f}%)")


Average WER: 0.0290 (2.90%)
Average CER: 0.0082 (0.82%)


In [ ]:
from cramer_ipm import cramer_ipm_attack

In [ ]:
# Load metrics
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

# Select example
example = librispeech[11]
audio_array = example["audio"]["array"]  # Raw audio waveform
ground_truth = example["true_text"]      # Ground truth transcription
target_transcription = "HELLO WORLD"     # Target transcription
# Run the Cramér-IPM attack
adversarial_waveform = cramer_ipm_attack(
    audio_array=audio_array,
    ground_truth=ground_truth,
    target_transcription=target_transcription,
    model=model,
    processor=processor,
    epsilon=0.001,
    num_iterations=10,
    lambda_ipm=1
)

original_transcription = transcribe_audio(audio_array, 16000, processor, model)
adversarial_transcription = transcribe_audio(adversarial_waveform, 16000, processor, model)


# Calculate CER and WER
cer_original = cer_metric.compute(predictions=[original_transcription], references=[ground_truth])
wer_original = wer_metric.compute(predictions=[original_transcription], references=[ground_truth])
cer = cer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])
wer = wer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])

# Display audio
print("Original Audio:")
display(ipd.Audio(audio_array, rate=16000))
print("Adversarial Audio:")
display(ipd.Audio(adversarial_waveform, rate=16000))

# Print transcription and metrics
print(f"Ground Truth: {ground_truth}")
print(f"Original Transcription: {original_transcription}")
print(f"Adversarial Transcription: {adversarial_transcription}")
print(f"Original CER: {cer_original:.2f}")
print(f"Original WER: {wer_original:.2f}")
print(f"Adversarial CER: {cer:.2f}")
print(f"Adversarial WER: {wer:.2f}")

Original Audio:


Adversarial Audio:


Ground Truth: AS USED IN THE SPEECH OF EVERYDAY LIFE THE WORD CARRIES AN UNDERTONE OF DEPRECATION
Original Transcription: AS USED IN THE SPEECH OF EVERYDAY LIFE THE WORD CARRIES AN UNDERTONE OF DEPRECATION
Adversarial Transcription: AS USIN A SPEECH OF EVERY DAY LIFE THE GOD CARIS AN UNDERTINATON
Original CER: 0.00
Original WER: 0.00
Adversarial CER: 0.29
Adversarial WER: 0.67


In [ ]:
# Select three sample indices for demonstration
selected_indices = [0, 1, 2]

# Define epsilon values to test
epsilon_values = [0.001, 0.01, 0.1]

# Set a fixed target transcription for the attack
target_transcription = "HELLO WORLD"

# Evaluate for each epsilon
for epsilon in epsilon_values:
    print(f"\n=== Epsilon: {epsilon} ===")

    # Lists to store metrics for all samples
    cer_list = []
    wer_list = []
    demo_samples = []

    # Process the entire dataset
    for idx, example in enumerate(librispeech):
        audio_array = example["audio"]["array"]
        ground_truth = example["true_text"]

        # Run the Cramér-IPM attack
        adversarial_waveform = cramer_ipm_attack(
            audio_array=audio_array,
            ground_truth=ground_truth,
            target_transcription=target_transcription,
            model=model,
            processor=processor,
            epsilon=epsilon,
            num_iterations=10,
            lambda_ipm=1
        )

        if idx % 10 == 0:
            print(f"Processed {idx} samples")

        # Transcribe original and adversarial audio
        original_transcription = transcribe_audio(audio_array, 16000, processor, model)
        adversarial_transcription = transcribe_audio(adversarial_waveform, 16000, processor, model)

        # Calculate CER and WER
        cer = cer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])
        wer = wer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])

        # Store metrics
        cer_list.append(cer)
        wer_list.append(wer)

        # Store info for demo samples
        if idx in selected_indices:
            demo_samples.append({
                'original_audio': audio_array,
                'adversarial_audio': adversarial_waveform,
                'original_transcription': original_transcription,
                'adversarial_transcription': adversarial_transcription,
                'ground_truth': ground_truth,
                'cer': cer,
                'wer': wer
            })

    # Compute and print overall metrics
    overall_cer = np.mean(cer_list)
    overall_wer = np.mean(wer_list)
    print(f"Overall CER: {overall_cer:.2f}")
    print(f"Overall WER: {overall_wer:.2f}")

    # Display detailed info and audio for the three samples
    i=0
    for sample in demo_samples:
        print(f"\nSample {i}:")
        i+=1
        print(f"Ground Truth: {sample['ground_truth']}")
        print(f"Original Transcription: {sample['original_transcription']}")
        print(f"Adversarial Transcription: {sample['adversarial_transcription']}")
        print(f"CER: {sample['cer']:.2f}")
        print(f"WER: {sample['wer']:.2f}")
        print("Original Audio:")
        display(ipd.Audio(sample['original_audio'], rate=16000))
        print("Adversarial Audio:")
        display(ipd.Audio(sample['adversarial_audio'], rate=16000))


=== Epsilon: 0.001 ===
Processed 0 samples
Processed 10 samples
Processed 20 samples
Processed 30 samples
Processed 40 samples
Processed 50 samples
Processed 60 samples
Processed 70 samples
Processed 80 samples
Overall CER: 0.22
Overall WER: 0.46

Sample 0:
Ground Truth: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Original Transcription: AND WHAT SORT OF EVIDENCE IS LOGICALLY POSSIBLE
Adversarial Transcription: IN WHAT SORT O EVIDENCE IS LOGICALLY POSSIBLE
CER: 0.06
WER: 0.25
Original Audio:


Adversarial Audio:



Sample 1:
Ground Truth: REMEMBERING HAS TO BE A PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
Original Transcription: REMEMBERING HAS TO BE A PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
Adversarial Transcription: WHO ON BIN HAS TO BE PROS DECURRENCE IN SOME WAY RESEMBLIN OLATED TO WHAT IS REMEMBERED
CER: 0.22
WER: 0.53
Original Audio:


Adversarial Audio:



Sample 2:
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Original Transcription: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: ALLRO THINK SUCH AN EFFERENCE AS WARN'T IT
CER: 0.41
WER: 0.80
Original Audio:


...

Adversarial Audio:



Sample 1:
Ground Truth: REMEMBERING HAS TO BE A PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
Original Transcription: REMEMBERING HAS TO BE A PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
Adversarial Transcription: R
CER: 0.99
WER: 1.00
Original Audio:


Adversarial Audio:



Sample 2:
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Original Transcription: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: HN
CER: 0.96
WER: 1.00
Original Audio:


Adversarial Audio:
